In [142]:
import json
import pandas as pd
import plotly.express as px

In [143]:
df = pd.read_csv("../data/processed/movies_clean.csv")
df.head()

,budget,revenue,vote_average,vote_count,imdbID,Title,Year,Genre,Metascore,imdbRating,studio_type,audience_score,metascore_10,gap
0,55000000,1078958629,8.122,27972,tt7286456,Joker,2019,"Crime, Drama, Thriller",59.0,8.3,major,8.2110,5.9,2.3110
1,150000000,378858340,7.635,24294,tt1392190,Mad Max: Fury Road,2015,"Action, Adventure, Sci-Fi",90.0,8.1,major,7.8675,9.0,-1.1325
2,175000000,749200054,5.917,22234,tt1386697,Suicide Squad,2016,"Action, Adventure, Fantasy",40.0,5.9,major,5.9085,4.0,1.9085
3,150000000,1671537444,6.703,21727,tt0369610,Jurassic World,2015,"Action, Adventure, Sci-Fi",59.0,6.9,major,6.8015,5.9,0.9015
4,149000000,823970682,7.207,21027,tt0451279,Wonder Woman,2017,"Action, Adventure, Fantasy",76.0,7.3,major,7.2535,7.6,-0.3465


# Part 1. Overall Comparison: Major vs Indie Films(Box plot)

To investigate whether major studio films exhibit larger audience–critic rating gaps than indie films, the distribution of `gap` values was first compared between the two studio groups.

The `gap` variable represents the difference between audience ratings and critic ratings:

A boxplot was used to compare the overall distribution of gaps between major and indie films. Individual observations were also displayed to show the variation among movies.


In [144]:
fig = px.box(df, x="studio_type", y="gap", points="all")
fig.show()

### Observation

💡The boxplot shows that major studio films have a **slightly higher** median audience–critic gap compared with indie films.

However, the two distributions overlap substantially, suggesting that the overall difference between major and indie films is **relatively modest**. Therefore, the result alone does not provide strong evidence that major studio affiliation consistently leads to higher audience ratings relative to critics.

💡An additional observation is that indie films exhibit a **wider spread of gap values**, indicating **greater variation** in the level of disagreement between audiences and critics. While some indie films receive audience ratings close to critic evaluations, others show much larger differences.

This suggests that studio type may influence the consistency of audience–critic agreement, although further analysis is needed to explore possible factors behind this variation.

## Gap Direction Analysis

The previous comparison examined the overall distribution of audience–critic gaps. To further understand where the difference comes from, the gap was separated into two directions:

- **Audience higher**: audience ratings are higher than critic ratings (`gap > 0`)
- **Expert higher**: critic ratings are higher than audience ratings (`gap < 0`)

This allows us to examine whether major and indie films differ in the direction of disagreement rather than only comparing the overall average gap.

In [145]:
df["gap_direction"] = df["gap"].apply(lambda x: "Audience higher" if x > 0 else "Expert higher")

fig = px.box(
    df,
    x="gap_direction",
    y="gap",
    color="studio_type",
    points="all",
    color_discrete_map={
        "Major": "#4C78A8",
        "Indie": "#F58518"
    }
)

fig.show()

### Observation

When audiences rate films higher than critics (`Audience higher`), the distributions of major and indie films are relatively similar, suggesting limited difference between the two studio groups in this direction.

However, when critics rate films higher than audiences (`Expert higher`), indie films show a larger gap compared with major studio films. The difference is approximately 0.3 rating points.

💡This suggests that indie films may experience stronger disagreement when professional critics evaluate them more positively than general audiences. In other words, the difference between major and indie films appears to be driven more by cases where audiences are less favourable than critics, rather than cases where audiences rate films more highly.

# Step 2 Distribution of Audience–Critic Gaps(histogram)

The boxplot provides a summary comparison of the gap distributions, but it does not show the full shape of the distributions.

Therefore, histograms were used to further examine how audience–critic gaps are distributed across major and indie films.

### Initial Distribution Comparison

The first histogram compares the distribution of gap values using the number of movies in each bin.


In [146]:
fig = px.histogram(
    df,
    x="gap",
    color="studio_type",
    barmode="overlay"
)
fig.show()

However, the two studio groups contain different numbers of movies. Therefore, comparing raw counts may be misleading because a larger sample group naturally produces higher frequencies.

To enable a fair comparison, the distribution was normalised by converting counts into percentages.

In [147]:
fig = px.histogram(
    df,
    x="gap",
    color="studio_type",
    histnorm="percent",
    barmode="overlay",
    opacity=0.4,
    color_discrete_map={
        "Major": "#4C78A8",
        "Indie": "#F58518"
    },
    labels={
        "gap": "Audience Score − Critic Score",
        "percent": "Percentage of Movies",
        "studio_type": "Studio Type"
    },
    title="Distribution of Rating Gap by Studio Type"
)

fig.update_layout(
    yaxis_title="Percentage of Movies"
)

fig.show()

Although percentage normalisation allows comparison between groups with different sample sizes, the overlapping distributions make it difficult to clearly compare the two groups.

Therefore, the distributions were separated into individual plots while keeping the same axis ranges to allow direct comparison.

### Separate into 2 comparable graph
The dataset was divided into major and indie groups to improve visual clarity.

The range of the x-axis and y-axis was kept consistent between plots to ensure that differences in distribution were not caused by different scales.

In [148]:
major_df = df[df["studio_type"] == "major"]
indie_df = df[df["studio_type"] == "indie"]

print(f"Major movies: {len(major_df)}")
print(f"Independent movies: {len(indie_df)}")

Major movies: 209
Independent movies: 668


In [149]:
df["gap"].describe()

count    849.000000
mean       0.808256
std        1.288724
min       -2.521500
25%       -0.049000
50%        0.817500
75%        1.681500
max        4.725000
Name: gap, dtype: float64

In [150]:
fig_major = px.histogram(
    major_df,
    x="gap",
    histnorm="percent",
    color_discrete_sequence=["#4C78A8"],
    labels={
        "gap": "Audience Score − Critic Score",
        "percent": "Percentage of Movies"
    },
    title="Major Studios"
)
fig_major.update_xaxes(range=[-3, 5])
fig_major.update_traces(
    xbins=dict(
        start=-3,
        end=5,
        size=0.2
    )
)
fig_major.show()

In [151]:
fig_indie = px.histogram(
    indie_df,
    x="gap",
    histnorm="percent",
    color_discrete_sequence=["#F58518"],
    labels={
        "gap": "Audience Score − Critic Score",
        "percent": "Percentage of Movies"
    },
    title="Independent Studios"
)
fig_indie.update_xaxes(range=[-3, 5])

fig_indie.update_traces(
    xbins=dict(
        start=-3,
        end=5,
        size=0.2
    )
)
fig_indie.show()

To improve interpretation, additional reference information was added:

- A vertical line at zero indicates where audience and critic ratings are equal.
- The mean gap shows the average level of disagreement for each group.
- The proportions on each side of zero indicate how often audiences rated films higher or lower than critics.

In [152]:
fig_major.update_yaxes(range=[0, 8])
fig_indie.update_yaxes(range=[0, 8])

fig_indie.show()
fig_major.show()

In [153]:
fig_major.add_shape(
    type="line",
    x0=0,
    x1=0,
    y0=0,
    y1=5.26,
    line=dict(
        color="black",
        width=1.5
    )
)

fig_indie.add_shape(
    type="line",
    x0=0,
    x1=0,
    y0=0,
    y1=5.1,
    line=dict(
        color="black",
        width=1.5
    )
)
fig_major.show()
fig_indie.show()

In [154]:
major_positive = (major_df["gap"] > 0).mean() * 100
major_negative = (major_df["gap"] <= 0).mean() * 100

indie_positive = (indie_df["gap"] > 0).mean() * 100
indie_negative = (indie_df["gap"] <= 0).mean() * 100

In [155]:
fig_major.add_annotation(
    x=-1.5,
    y=7.5,
    text=f"Critics higher<br>{major_negative:.1f}%",
    showarrow=False
)

fig_major.add_annotation(
    x=3,
    y=7.5,
    text=f"Audience higher<br>{major_positive:.1f}%",
    showarrow=False
)


fig_indie.add_annotation(
    x=-1.5,
    y=7.5,
    text=f"Critics higher<br>{indie_negative:.1f}%",
    showarrow=False
)

fig_indie.add_annotation(
    x=3,
    y=7.5,
    text=f"Audience higher<br>{indie_positive:.1f}%",
    showarrow=False
)
fig_major.show()

In [156]:
major_mean = major_df["gap"].mean()
indie_mean = indie_df["gap"].mean()

In [157]:
fig_major.add_vline(
    x=major_mean,
    line_dash="dash",
    line_color="red",
    annotation_text=f"Mean = {major_mean:.2f}",
    annotation_position="top"
)

fig_indie.add_vline(
    x=indie_mean,
    line_dash="dash",
    line_color="red",
    annotation_text=f"Mean = {indie_mean:.2f}",
    annotation_position="top"
)
fig_major.show()
fig_indie.show()

### Observation

The distributions show that both major and indie films include cases where audiences rate films higher than critics, as well as cases where critics rate films higher than audiences.

💡The overall distributions are relatively similar, but some differences can still be observed. 
💡Indie films have a higher proportion of negative gaps, where critics rate films higher than audiences, compared with major studio films (approximately 7% higher). This suggests that audience and critic opinions may diverge more frequently for indie films in this direction.

💡In addition, indie films show a wider distribution of gap values, including more extreme cases on both sides of the scale. Some indie films receive substantially higher audience ratings than critic ratings, while others receive substantially lower audience ratings. This indicates greater variability in audience–critic disagreement among indie films.

💡Overall, the histogram analysis suggests that studio type may be related to the pattern of audience–critic disagreement. However, the difference is not large enough to conclude that major studio affiliation alone creates a strong rating advantage.

# Part 3: Explore whether the audience–critic gap shows a temporal trend (line plot)

After examining differences between major and indie films, a further question is whether the audience–critic gap has changed over time.

With the growth of social media, online communities, and user-generated reviews, audience opinions have become increasingly visible and influential in the film industry. This raises the possibility that the gap between audience evaluations and professional critics' evaluations may have changed in recent years.


In [158]:
year_gap = (
    df
    .groupby(["Year", "studio_type"])["gap"]
    .mean()
    .reset_index()
)

In [159]:
fig = px.line(
    year_gap,
    x="Year",
    y="gap",
    color="studio_type",
    markers=True,
    color_discrete_map={
        "major": "#4C78A8",
        "indie": "#F58518"
    }
)
fig.show()

### Why 2014???
Two movies were labelled as 2014 in OMDb but as 2015 releases in TMDB. Since this analysis uses OMDb year values, these two records were removed to keep the analysis period consistent with 2015–2024.

This highlights a limitation when integrating data from multiple APIs, where the same attribute may differ across sources.

In [160]:
df[df["Year"] == 2014]

,budget,revenue,vote_average,vote_count,imdbID,Title,Year,Genre,Metascore,imdbRating,studio_type,audience_score,metascore_10,gap,gap_direction
251,15000000,36869414,7.571,14404,tt0470752,Ex Machina,2014,"Drama, Sci-Fi, Thriller",78.0,7.7,indie,7.6355,7.8,-0.1645,Expert higher
364,2300000,23374076,6.583,7021,tt3235888,It Follows,2014,"Horror, Mystery, Thriller",83.0,6.8,indie,6.6915,8.3,-1.6085,Expert higher


In [161]:
year_counts = (
    df
    .groupby(["Year", "studio_type"])
    .size()
    .reset_index(name="count")
)

year_counts

,Year,studio_type,count
0,2014,indie,2
1,2015,indie,78
2,2015,major,29
3,2016,indie,107
4,2016,major,26
5,2017,indie,82
6,2017,major,25
7,2018,indie,81
8,2018,major,32
9,2019,indie,89


In [162]:
year_2015 = df[df["Year"] >= 2015]
year_2015_gap = (
    year_2015
    .groupby(["Year", "studio_type"])["gap"]
    .mean()
    .reset_index()
)
fig = px.line(
    year_2015_gap,
    x="Year",
    y="gap",
    color="studio_type",
    markers=True,
    color_discrete_map={
        "major": "#4C78A8",
        "indie": "#F58518"
    }
)
fig.show()

### Observation

The yearly trend does not show a clear increasing or decreasing pattern in audience–critic gaps over time.

Although major and indie films appear to fluctuate in similar directions in some years, there is no consistent evidence that the disagreement between audiences and critics has systematically increased in recent years.

This suggests that factors such as the growth of online platforms and social media alone may not explain changes in audience–critic differences.

However, yearly sample sizes vary, and some years contain fewer observations, so these results should be interpreted cautiously.

# Part 4：Genre-level Analysis (bar plot)
The previous analysis examined differences between major and indie films at the overall level. However, audience–critic disagreement may vary depending on movie genre.

Therefore, the data was further explored at the genre level to investigate whether certain genres show larger differences between major and indie films.

## Expanding genres
Each movie may belong to multiple genres. Therefore, genre information was expanded so that each movie–genre combination was treated as an individual observation.

In [163]:
df_genre = df.copy()

df_genre["genre_type"] = df_genre["Genre"].str.split(",")

df_genre = df_genre.explode("genre_type")

df_genre["genre_type"] = df_genre["genre_type"].str.strip()

In [164]:
genre_gap_df = (
    df_genre
    .groupby(["genre_type","studio_type"])["gap"]
    .mean()
    .reset_index()
)


In [165]:
fig = px.bar(
    genre_gap_df,
    x="gap",
    y="genre_type",
    color="studio_type",
    barmode="group",
    orientation="h",
    color_discrete_map={
        "Major": "#4C78A8",
        "Indie": "#F58518"
    }
)

fig.show()

### Detect an issue: missing columns
Before comparing genres, the number of observations in each genre and studio group forgot to examined.

Genres with insufficient observations were removed to avoid drawing conclusions from very small samples.

In [166]:
genre_counts = df_genre.groupby(["genre_type", "studio_type"]).size().reset_index(name="count")
genre_counts

,genre_type,studio_type,count
0,Action,indie,263
1,Action,major,95
2,Adventure,indie,220
3,Adventure,major,100
4,Animation,indie,69
5,Animation,major,22
6,Biography,indie,47
7,Biography,major,15
8,Comedy,indie,205
9,Comedy,major,67


The minimum number of observations between major and indie groups was used as the filtering criterion. Only genres with at least 20 movies in both groups were retained.


In [167]:
valid_genres = (
    genre_counts
    .groupby(["genre_type"])["count"]
    .min()
    .reset_index() #otherwise its a seires, instead of a dataframe
    .query("count >= 20")
)
valid_genres


,genre_type,count
0,Action,95
1,Adventure,100
2,Animation,22
4,Comedy,67
5,Crime,26
7,Drama,70
9,Fantasy,32
11,Horror,35
14,Mystery,24
16,Sci-Fi,29


In [168]:
# Keep only genres that satisfy the minimum sample size requirement
df_genre = df_genre[
    df_genre["genre_type"].isin(valid_genres["genre_type"])
]

### Calculate average by genre
For each remaining genre, the average audience–critic gap was calculated separately for major and indie films.

In [169]:
genre_gap_df = (
    df_genre
    .groupby(["genre_type", "studio_type"])["gap"]
    .mean()
    .reset_index()
)
genre_gap_df

,genre_type,studio_type,gap
0,Action,indie,1.162045
1,Action,major,1.143968
2,Adventure,indie,1.002687
3,Adventure,major,1.164320
4,Animation,indie,0.692529
5,Animation,major,1.180250
6,Comedy,indie,0.790425
7,Comedy,major,1.153493
8,Crime,indie,0.808461
9,Crime,major,0.973577


In [170]:
fig = px.bar(
    genre_gap_df,
    x="gap",
    y="genre_type",
    color="studio_type",
    orientation="h",
    barmode="group",
    color_discrete_map={
        "Major": "#4C78A8",
        "Indie": "#F58518"
    }
)

fig.show()

### Small tweak:
To make the differences easier to interpret, genres were ranked according to the absolute difference between major and indie films.

A larger absolute difference indicates a larger difference in audience–critic gap between the two studio groups, regardless of direction.

In [171]:
genre_difference = (
    genre_gap_df
    .groupby("genre_type")["gap"]
    .agg(lambda x: x.iloc[1] - x.iloc[0]) #major-indie
    .reset_index(name="difference")
)

genre_difference

,genre_type,difference
0,Action,-0.018077
1,Adventure,0.161633
2,Animation,0.487721
3,Comedy,0.363067
4,Crime,0.165116
5,Drama,0.249973
6,Fantasy,0.238750
7,Horror,0.072677
8,Mystery,-0.330691
9,Sci-Fi,-0.207318


In [172]:
genre_difference["abs_difference"] = genre_difference["difference"].abs()

In [173]:
genre_order = (
    genre_difference
    .sort_values("abs_difference", ascending=False)
    ["genre_type"]
    .tolist()
)
genre_order

['Animation',
 'Comedy',
 'Mystery',
 'Thriller',
 'Drama',
 'Fantasy',
 'Sci-Fi',
 'Crime',
 'Adventure',
 'Horror',
 'Action']

In [174]:
fig = px.bar(
    genre_gap_df,
    x="gap",
    y="genre_type",
    color="studio_type",
    orientation="h",
    barmode="group",
    color_discrete_map={
        "Major": "#4C78A8",
        "Indie": "#F58518"
    },
        category_orders={
        "genre_type": genre_order
    },
)

fig.show()

To directly examine which genres contribute most to the overall difference, the absolute Major–Indie gap difference was visualized.

Positive and negative directions were separated to indicate whether the larger gap occurs among major or indie films.

In [175]:
genre_difference["direction"] = genre_difference["difference"].apply(
    lambda x: "Bigger gap in major" if x > 0 else "Bigger gap in indie"
)

In [176]:
fig = px.bar(
    genre_difference,
    x="abs_difference",
    y="genre_type",
    color="direction",
    orientation="h",
    color_discrete_map={
        "Bigger gap in major": "#4C78A8",
        "Bigger gap in indie": "#F58518"
    },
    category_orders={
        "genre_type": genre_order
    } 
)

fig.update_layout(
    title="Difference in Audience–Critic Gap Between Major and Indie Films by Genre",
    xaxis_title="Absolute Difference in Gap",
    yaxis_title="Genre"
)

fig.show()

### Observation

The genre-level analysis shows that differences between major and indie films are not consistent across all genres.

💡Some genres show relatively small differences, while others exhibit relatively larger differences in audience–critic gaps. This suggests that genre characteristics may influence the level of disagreement between audiences and critics.

Among the examined genres, 💡Animation shows one of the largest differences, with indie films exhibiting a larger audience–critic gap than major studio films. In contrast, 💡Mystery shows the largest difference in the opposite direction, where major studio films have a larger gap.

These findings suggest that the relationship between studio type and audience–critic disagreement may depend on genre rather than following a single overall pattern.

# Part 5: Further Exploration

After identifying notable differences across genres, additional variables were explored to investigate possible explanations for the observed patterns.

### Budget Analysis

The relationship between production budget and audience–critic gap was examined to explore whether financial investment could explain differences between major and indie films.

No clear relationship was observed between budget and gap, suggesting that higher production budgets alone do not explain differences in audience–critic disagreement.

### Genre-specific Exploration

Further analysis was conducted within specific genres, particularly Animation, which showed one of the largest Major–Indie differences.

However, additional exploration did not identify a clear factor that consistently explains the observed gap differences.

Overall, these exploratory analyses suggest that while some differences exist between major and indie films, the observed patterns are likely influenced by multiple factors rather than a single explanatory variable.

In [177]:
animation_df = df_genre[
    df_genre["genre_type"] == "Animation"
].copy()
animation_df.head()

,budget,revenue,vote_average,vote_count,imdbID,Title,Year,Genre,Metascore,imdbRating,studio_type,audience_score,metascore_10,gap,gap_direction,genre_type
24,74000000,1159457503,6.421,11184,tt2293640,Minions,2015,"Animation, Adventure, Comedy",56.0,6.4,major,6.4105,5.6,0.8105,Audience higher,Animation
26,100000000,1360879735,7.602,10934,tt6718170,The Super Mario Bros. Movie,2023,"Animation, Adventure, Comedy",46.0,7.0,major,7.3010,4.6,2.7010,Audience higher,Animation
29,260000000,1662020819,7.095,10814,tt6105098,The Lion King,2019,"Animation, Adventure, Drama",55.0,6.8,major,6.9475,5.5,1.4475,Audience higher,Animation
44,75000000,875457937,6.300,8567,tt2709768,The Secret Life of Pets,2016,"Animation, Adventure, Comedy",61.0,6.5,major,6.4000,6.1,0.3000,Audience higher,Animation
46,175000000,966550600,6.877,8558,tt3040964,The Jungle Book,2016,"Animation, Action, Adventure",77.0,7.3,major,7.0885,7.7,-0.6115,Expert higher,Animation


In [178]:
fig_budget = px.scatter(
    animation_df,
    x="budget",
    y="gap",
    color="studio_type",
    hover_name="Title",
    color_discrete_map={
        "major": "#4C78A8",
        "indie": "#F58518"
    }
)

fig_budget.update_layout(
    title="Budget vs Audience–Critic Gap for Animation Films",
    xaxis_title="Budget",
    yaxis_title="Audience–Critic Gap"
)

fig_budget.show()

In [179]:
fig_budget.update_xaxes(range=[0, 200000000])
fig_budget.show()

In [180]:
fig = px.scatter(
    animation_df,
    x="budget",
    y="gap",
    color="studio_type",
    trendline="ols",
    hover_name="Title",
    color_discrete_map={
        "major": "#4C78A8",
        "indie": "#F58518"
    }
)

fig.show()

In [181]:
fig = px.box(
    animation_df,
    x="studio_type",
    y="budget",
    color="studio_type",
    points="all",
    color_discrete_map={
        "major": "#4C78A8",
        "indie": "#F58518"
    }
)

fig.update_layout(
    title="Budget Distribution of Major and Indie Animation Films",
    xaxis_title="Studio Type",
    yaxis_title="Budget"
)

fig.show()

In [182]:
comedy_df = df_genre[
    df_genre["genre_type"] == "Comedy"
].copy()
comedy_df.head()

,budget,revenue,vote_average,vote_count,imdbID,Title,Year,Genre,Metascore,imdbRating,studio_type,audience_score,metascore_10,gap,gap_direction,genre_type
21,145000000,1447138421,6.916,11292,tt1517268,Barbie,2023,"Adventure, Comedy, Fantasy",80.0,6.8,major,6.8580,8.0,-1.1420,Expert higher,Comedy
24,74000000,1159457503,6.421,11184,tt2293640,Minions,2015,"Animation, Adventure, Comedy",56.0,6.4,major,6.4105,5.6,0.8105,Audience higher,Comedy
26,100000000,1360879735,7.602,10934,tt6718170,The Super Mario Bros. Movie,2023,"Animation, Adventure, Comedy",46.0,7.0,major,7.3010,4.6,2.7010,Audience higher,Comedy
27,75000000,205537933,6.870,10890,tt7713068,Birds of Prey and the Fantabulous Emancipation...,2020,"Action, Comedy, Crime",60.0,6.1,major,6.4850,6.0,0.4850,Audience higher,Comedy
30,183000000,1054304000,7.081,10705,tt6139732,Aladdin,2019,"Adventure, Comedy, Family",53.0,6.9,major,6.9905,5.3,1.6905,Audience higher,Comedy


In [183]:
fig_budget = px.scatter(
    comedy_df,
    x="budget",
    y="gap",
    color="studio_type",
    hover_name="Title",
    color_discrete_map={
        "major": "#4C78A8",
        "indie": "#F58518"
    }
)

fig_budget.update_layout(
    title="Budget vs Audience–Critic Gap for Comedy Films",
    xaxis_title="Budget",
    yaxis_title="Audience–Critic Gap"
)

fig_budget.show()

In [184]:
fig = px.box(
    comedy_df,
    x="studio_type",
    y="budget",
    color="studio_type",
    points="all",
    color_discrete_map={
        "major": "#4C78A8",
        "indie": "#F58518"
    }
)

fig.update_layout(
    title="Budget Distribution of Major and Indie comedy Films",
    xaxis_title="Studio Type",
    yaxis_title="Budget"
)

fig.show()

In [185]:
animation_year_gap = (
    animation_df
    .groupby(["Year", "studio_type"])["gap"]
    .mean()
    .reset_index()
)

In [186]:
fig = px.line(
    animation_year_gap,
    x="Year",
    y="gap",
    color="studio_type",
    markers=True,
    color_discrete_map={
        "major": "#4C78A8",
        "indie": "#F58518"
    }
)

fig.update_layout(
    title="Average Audience–Critic Gap of Animation Films Over Time",
    xaxis_title="Year",
    yaxis_title="Average Gap"
)

fig.show()

In [187]:
comedy_df.groupby(["Year","studio_type"]).size()

Year  studio_type
2015  indie          24
      major          13
2016  indie          35
      major           7
2017  indie          24
      major           7
2018  indie          26
      major           7
2019  indie          30
      major           4
2020  indie          12
      major           6
2021  indie          18
      major           6
2022  indie          18
      major           5
2023  indie          10
      major           7
2024  indie           8
      major           5
dtype: int64

In [188]:
comedy_year_gap = (
    comedy_df
    .groupby(["Year", "studio_type"])["gap"]
    .mean()
    .reset_index()
)

In [189]:
fig = px.line(
    comedy_year_gap,
    x="Year",
    y="gap",
    color="studio_type",
    markers=True,
    color_discrete_map={
        "major": "#4C78A8",
        "indie": "#F58518"
    }
)

fig.update_layout(
    title="Average Audience–Critic Gap of Comedy Films Over Time",
    xaxis_title="Year",
    yaxis_title="Average Gap"
)

fig.show()

# Discussion and Conclusion

## Key Findings

This project investigated whether major studio films exhibit a larger audience–critic rating gap compared with indie films, exploring the possibility of a potential "studio halo effect".

Overall, the results provide **some indication** that major studio films may experience a slightly larger audience–critic gap, which is consistent with the possibility of a studio reputation effect. **However**, the difference is relatively small and varies across genres, suggesting that studio affiliation alone is unlikely to fully explain audience–critic disagreement.

### 1. Overall Audience–Critic Gap

💡The overall comparison shows that major studio films have a slightly higher average audience–critic gap than indie films. However, the distributions overlap substantially, suggesting that studio type alone does not create a strong difference in audience–critic disagreement.

💡An additional observation is that indie films exhibit greater variation in gap values, indicating that audience and critic opinions are more dispersed for indie films.

### 2. Direction of Disagreement

When separating gaps by direction（+/-）, the difference between major and indie films becomes more informative.

For cases where audiences rate films higher than critics, the two studio groups show relatively similar patterns.

💡However, when critics rate films higher than audiences, indie films show a larger proportion of negative gaps compared with major films. This suggests that indie films may experience stronger disagreement in cases where professional critics evaluate them more positively than general audiences.

### 3. Genre Differences

The genre-level analysis suggests that the relationship between studio type and audience–critic disagreement is not uniform across genres.

💡Some genres show relatively small differences, while others exhibit substantial differences. Animation shows one of the largest differences, with indie films having a larger gap, while Mystery shows a large difference in the opposite direction, with major films having a larger gap.

This suggests that genre characteristics may play an important role in shaping audience–critic disagreement.

### 4. Additional Exploration

Additional exploration was conducted to investigate possible explanations for the observed patterns.

Neither production budget nor yearly trends showed a clear relationship with audience–critic gaps. This suggests that factors such as financial investment or changes over time alone are unlikely to fully explain the observed differences.

---

# Limitations

Despite providing insights into audience–critic disagreement, this analysis has several limitations.

### 1. Critic Score as a Proxy for Film Quality

Metascore was used as a reference for professional evaluation. However, critic scores do not represent an objective measure of film quality. Critics and audiences evaluate films based on different perspectives, preferences, and expectations.

Therefore, the calculated gap should be interpreted as a difference between two evaluation systems rather than a measure of absolute quality.

### 2. Studio Classification

The classification of major and indie films was simplified based on selected major production companies. Some films may involve multiple production partners or hybrid financing structures that do not fit perfectly into either category.

### 3. Sample Distribution

The dataset contains more indie films than major studio films, and some genres contain fewer observations. Although genres with insufficient sample sizes were removed, remaining differences should still be interpreted carefully.

### 4. Correlation Does Not Imply Causation

This study identifies patterns associated with studio type but cannot determine whether studio reputation directly causes differences in audience ratings.

Other factors, such as marketing, franchise status, star power, distribution strategy, or audience expectations may also influence rating differences.

---

# Conclusion

The findings suggest that major studio affiliation may be associated with slightly different audience–critic rating patterns, but the evidence for a strong studio halo effect is limited.

Instead, audience–critic disagreement appears to vary substantially depending on genre and individual film characteristics.

Therefore, rather than a simple major-versus-indie divide, the relationship between studio type and audience perception is likely influenced by multiple interacting factors.

## Notebook Summary

| Notebook | Main Purpose |
|----------|--------------|
| **NB01a** | Collect raw movie data from the TMDB API. |
| **NB01b** | Retrieve OMDb data using IMDb IDs collected from TMDB. |
| **NB02a** | Process and clean TMDB raw JSON into a structured dataset. |
| **NB02b** | Process OMDb data, merge both datasets, and construct the final analysis dataset through feature engineering. |
| **NB03** | Explore the data through visualisation, interpret findings, discuss limitations, and answer the research question. |